PageIndex RAG -> Build tree -> LLM reasons over tree -> retrieve exact sections


In [25]:
!pip install -U groq python-dotenv


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\subha\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [26]:
import os
from dotenv import load_dotenv
load_dotenv()

PAGEINDEX_API_KEY=os.getenv("PAGEINDEX_API_KEY")
GROQ_API_KEY=os.getenv("GROQ_API_KEY")


In [27]:
from pageindex import PageIndexClient
from groq import Groq

pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)
groq_client = Groq(api_key=GROQ_API_KEY)

NOW section 2 is upload and index a pdf

What happens here:
Upload your pdf to the pageindex cloud
pageindex uses LLM to read the document structure
build the hierarchical tree iindex like a smart table of contents
returns a doc_id for all future operations

Why no chunking?
Instead of cutting the doc into arbitrary 500 token pieces PageIndex respects the document's natural section boundaries chapters sub-sections ,paragraphs as the author intended.

In [28]:
PDF_PATH="./data/pdf/attention.pdf"
print("Uploading PDF to PageIndex...")
result=pi_client.submit_document(PDF_PATH)
doc_id=result["doc_id"]
print(f"Document uploaded successfully. Document ID: {doc_id}")


Uploading PDF to PageIndex...
Document uploaded successfully. Document ID: pi-cmr94n2d100x301o3to9j3426


PageIndex builds the tree asynchronously
for a 50 page PDF this is typically 30-90 seconds

In [30]:
from time import time


print("Building the index...")

while True:
    status_result=pi_client.get_document(doc_id)
    status=status_result.get("status")
    
    if status=="completed":
        print("Index built successfully.")
        break
    elif status=="failed":
        print("Index building failed.")
        break
   
       
    time.sleep(5)

Building the index...
Index built successfully.


Section 3 is Inspect the Tree Structure


Document
|__Introduction
     |__Background
|__FInancial Stability..
..
..
Each Node has:
node_id- Unique id used during retrieval
title - section heading
page_index- page number in original pdf
text- section summary(when node_summary=True)
nodes-child sections(nested)

THis structure is what the LLM reasons over during interval

In [31]:
import json

In [32]:
tree_result=pi_client.get_tree(doc_id,node_summary=True)
pageindex_tree=tree_result.get("result",[])

print(f"Top level sections: {len(pageindex_tree)}")
print("Raw tree(first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else None, indent=2))

Top level sections: 1
Raw tree(first node):
{
  "title": "Attention Is All You Need",
  "node_id": "0000",
  "page_index": 1,
  "prefix_summary": "# Attention Is All You Need\n\n**Ashish Vaswani***\nGoogle Brain\navaswani@google.com\n\n**Noam Shazeer***\nGoogle Brain\nnoam@google.com\n\n**Niki Parmar***\nGoogle Research\nnikip@google.com\n\n**Jakob Uszkoreit***\nGoogle Research\nusz@google.com\n\n**Llion Jones***\nGoogle Research\nllion@google.com\n\n**Aidan N. Gomez*** \u2020\nUniversity of Toronto\naidan@cs.toronto.edu\n\n**\u0141ukasz Kaiser***\nGoogle Brain\nlukaszkaiser@google.com\n\n**Illia Polosukhin*** \u2021\nillia.polosukhin@gmail.com\n",
  "text": "# Attention Is All You Need\n\n**Ashish Vaswani***\nGoogle Brain\navaswani@google.com\n\n**Noam Shazeer***\nGoogle Brain\nnoam@google.com\n\n**Niki Parmar***\nGoogle Research\nnikip@google.com\n\n**Jakob Uszkoreit***\nGoogle Research\nusz@google.com\n\n**Llion Jones***\nGoogle Research\nllion@google.com\n\n**Aidan N. Gomez*** \u20

Section 4 is LLM Tree Search- THe core of pageindex

In [33]:
def llm_tree_search(query:str,tree:list,model:str="llama-3.3-70b-versatile")->dict:
    """
    Perform a tree search using the provided query and tree structure.
    
    Args:
        query (str): The search query.
        tree (list): The tree structure to search within.
        model (str): The language model to use for the search. Default is "gpt-4o".
        
    Returns:
        dict: The search result containing relevant information.
    """
    # Implement the logic for tree search using the specified model
    
    def compress(nodes):
        out=[]
        for n in nodes:
            entry={
                "node_id":n["node_id"],
                "title":n["title"],
                "page":n.get("page_index","?"),
                "summary":n.get("text","")[:150]
            }
            if n.get("nodes"):
                entry["children"]=compress(n["nodes"])
            out.append(entry)
        return out
    
    dompressed_tree=compress(tree)
    
    prompt=f"""
    you are given a query and a document's trees strcuture .
    Your task: Identify which nodes in the tree are most relevant to the query.
    THink step by step about which sections are relevant.
    
    Query:{query}
    
    Document Tree:
    {json.dumps(dompressed_tree,indent=2)}
    
    Reply only in this exact JSON format:
    {{
        "thinking":"<your reasoning about which nodes are relevant>",
        "relevant_nodes":["node_id_1","node_id_2",...]
    }}


   """
   
    response=groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role":"user","content":prompt}],
        response_format={"type":"json_object"}
   )
    return response.choices[0].message

In [34]:
query="What is the main idea of the paper?"
search_result=llm_tree_search(query,pageindex_tree)

# llm_tree_search returns a ChatCompletionMessage; convert its JSON content to dict
search_result = json.loads(search_result.content)

print("Search Result:")
print(search_result.get("thinking","N/A"))

print("Relevant Nodes:")
print(search_result.get("relevant_nodes",[]))

Search Result:
To find the main idea of the paper, we should look at the abstract and introduction sections, as they typically provide an overview of the paper's content. Additionally, the conclusion section often summarizes the main points and contributions of the paper. Based on the document tree, the relevant nodes are likely to be '0001' (Abstract), '0002' (1 Introduction), and '0019' (7 Conclusion).
Relevant Nodes:
['0001', '0002', '0019']
